In [ ]:
!pip install numpy==1.26.4 pandas==2.2.2 pyarrow==15.0.2 "datasets==2.20.0" --force-reinstall --quiet

In [1]:
# Static validation: check the relative magnitudes of the score-matching loss and the
# physics term on a synthetic batch BEFORE committing to a full Kaggle training run.
# This avoids the lambda-scaling failure mode (Hypothesis 3).
import torch

torch.manual_seed(0)
B, C, F, T = 4, 1, 256, 256                # SGMSE+ STFT shape with n_fft=510, num_frames=256
sigma_val = 0.5                            # representative diffusion noise level

# Synthesize a smooth clean spectrogram + noisy x_t
freq = torch.linspace(0, 1, F).view(1, 1, F, 1)
time = torch.linspace(0, 1, T).view(1, 1, 1, T)
x_clean = (torch.exp(-freq * 4.0) * torch.cos(2 * 3.14159 * time * 3)).expand(B, C, F, T)
x_clean = torch.complex(x_clean, 0.5 * x_clean.roll(1, dims=-1))
z       = torch.randn_like(x_clean.real) + 1j * torch.randn_like(x_clean.real)
x_t     = x_clean + sigma_val * z
# A perturbed "score" estimate, of the same scale a trained model would produce.
score   = -z / sigma_val + 0.1 * (torch.randn_like(z.real) + 1j * torch.randn_like(z.real))

# (1) Standard score-matching loss: mean over batch of 0.5 * sum |score*sigma + z|^2
sm = torch.square(torch.abs(score * sigma_val + z))
loss_sm = torch.mean(0.5 * torch.sum(sm.reshape(B, -1), dim=-1))

# (2) Physics term: spectral envelope smoothness on log-magnitude of Tweedie x_0_hat
x_hat_spec = x_t + (sigma_val ** 2) * score
mag = torch.abs(x_hat_spec).clamp(min=1e-7)
log_mag = torch.log(mag)
d2_f = log_mag[:, :, 2:, :] - 2.0 * log_mag[:, :, 1:-1, :] + log_mag[:, :, :-2, :]
loss_phys = torch.mean(d2_f ** 2)

print(f'score-matching loss : {loss_sm.item():.4e}')
print(f'physics loss (raw)  : {loss_phys.item():.4e}')
print(f'ratio  phys / sm    : {(loss_phys.item() / loss_sm.item()):.4e}')

# Choose physics_weight so the physics contribution is ~1-5% of the SM loss at init.
target_fraction = 0.02
suggested_w = target_fraction * loss_sm.item() / loss_phys.item()
print(f'suggested physics_weight (for ~{target_fraction*100:.0f}% contribution): {suggested_w:.4e}')

# NaN / inf sanity check on the finite difference path.
assert torch.isfinite(loss_phys), 'physics loss is non-finite!'
assert torch.isfinite(loss_sm),   'score-matching loss is non-finite!'
print('OK: both loss components are finite.')

score-matching loss : 1.6358e+02
physics loss (raw)  : 1.0686e+00
ratio  phys / sm    : 6.5325e-03
suggested physics_weight (for ~2% contribution): 3.0616e+00
OK: both loss components are finite.


In [2]:
# Clone sgmse. Pilot does NOT download the pretrained checkpoint — we train from random init.
import os, shutil
if os.path.exists('/kaggle/working/sgmse'):
    shutil.rmtree('/kaggle/working/sgmse')
os.chdir('/kaggle/working')
get_ipython().system('git clone https://github.com/sp-uhh/sgmse.git')
os.chdir('/kaggle/working/sgmse')
get_ipython().system('pip install -r requirements.txt --quiet')
get_ipython().system('pip install pesq pystoi pandas gdown --quiet')

Cloning into 'sgmse'...
remote: Enumerating objects: 1011, done.
remote: Counting objects: 100% (357/357), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 1011 (delta 276), reused 214 (delta 212), pack-reused 654 (from 3)
Receiving objects: 100% (1011/1011), 3.74 MiB | 29.00 MiB/s, done.
Resolving deltas: 100% (547/547), done.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires typeguard<5,>=4, but you have typeguard 2.13.3 which is incompatible.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.


In [3]:
# Patch model.py: add spectral-envelope smoothness term inside the score_matching branch.
# We compute a Tweedie estimate of x_0 from the score, then penalize the second
# difference of log|x_0_hat| along the frequency axis.
patch = '''
            # === physics-informed regularizer (spectral envelope smoothness) ===
            # Tweedie estimate of the clean spectrogram from the score.
            # For OUVE-SDE this is an approximation that ignores the drift term;
            # used here as a regularization target, not an exact reconstruction.
            x_hat_spec = x_t + (sigma ** 2) * score
            mag = torch.abs(x_hat_spec).clamp(min=1e-7)
            log_mag = torch.log(mag)
            # Second difference along the frequency axis (axis=2 of (B,C,F,T)).
            d2_f = log_mag[:, :, 2:, :] - 2.0 * log_mag[:, :, 1:-1, :] + log_mag[:, :, :-2, :]
            phys_loss = torch.mean(d2_f ** 2)
            loss = loss + self.physics_weight * phys_loss
'''

with open('/kaggle/working/sgmse/sgmse/model.py', 'r') as f:
    content = f.read()

# Insert physics_weight attribute in __init__ (after self.loss_type = loss_type).
content = content.replace(
    'self.loss_type = loss_type\n',
    'self.loss_type = loss_type\n        self.physics_weight = 0.0\n',
    1,
)

# Insert the physics term at the end of the score_matching branch — right before the
# `elif self.loss_type == "denoiser":` line.
old = '            loss = torch.mean(0.5*torch.sum(losses.reshape(losses.shape[0], -1), dim=-1))\n        elif self.loss_type == "denoiser":'
new = '            loss = torch.mean(0.5*torch.sum(losses.reshape(losses.shape[0], -1), dim=-1))\n' + patch + '        elif self.loss_type == "denoiser":'
assert old in content, 'Anchor for score_matching patch not found'
content = content.replace(old, new, 1)

with open('/kaggle/working/sgmse/sgmse/model.py', 'w') as f:
    f.write(content)

get_ipython().system('grep -n "physics_weight\|phys_loss\|x_hat_spec" /kaggle/working/sgmse/sgmse/model.py')

72:        self.physics_weight = 0.0
153:            x_hat_spec = x_t + (sigma ** 2) * score
154:            mag = torch.abs(x_hat_spec).clamp(min=1e-7)
158:            phys_loss = torch.mean(d2_f ** 2)
159:            loss = loss + self.physics_weight * phys_loss


<>:38: SyntaxWarning: invalid escape sequence '\|'
<>:38: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_146/3285149017.py:38: SyntaxWarning: invalid escape sequence '\|'
  get_ipython().system('grep -n "physics_weight\|phys_loss\|x_hat_spec" /kaggle/working/sgmse/sgmse/model.py')


In [4]:
import soundfile as sf
import numpy as np
from datasets import load_dataset, Audio

# Step 2 control: 3000 train / 100 valid / full 826 test (same test set as the
# original fine-tune notebooks so metrics are comparable).
TEST_DIR = "data/test"
TRAIN_DIR = "data/train"
VALID_DIR = "data/valid"
for d in (TEST_DIR, TRAIN_DIR, VALID_DIR):
    os.makedirs(f"{d}/clean", exist_ok=True)
    os.makedirs(f"{d}/noisy", exist_ok=True)

print("Loading full test set...")
test_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="test")
test_data = test_data.cast_column("clean", Audio(sampling_rate=16000))
test_data = test_data.cast_column("noisy", Audio(sampling_rate=16000))
for i, sample in enumerate(test_data):
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TEST_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TEST_DIR}/noisy/{fname}", noisy, 16000)
print(f"Wrote {len(test_data)} test samples")

print("Loading train set (3000 samples)...")
train_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="train")
train_data = train_data.cast_column("clean", Audio(sampling_rate=16000))
train_data = train_data.cast_column("noisy", Audio(sampling_rate=16000))
for i in range(3000):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TRAIN_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TRAIN_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 3000 train samples")

print("Writing validation set (100 samples)...")
for i in range(3000, 3100):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{VALID_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{VALID_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 100 validation samples")

Loading full test set...


Generating train split:   0%|          | 0/11572 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/824 [00:00<?, ? examples/s]

Wrote 824 test samples
Loading train set (3000 samples)...
Wrote 3000 train samples
Writing validation set (100 samples)...
Wrote 100 validation samples


In [5]:
# Step 2 CONTROL training: random-init NCSNpp, MSE-only loss (physics_weight=0.0).
# This is one of the two paired runs that form the headline comparison.
# Identical to the experimental run except for physics_weight.
#
# Config:
#   - nf=32, batch_size=16, lr=1e-4 (cosine decay to 1e-5)
#   - 3000 train / 100 valid / 826 test
#   - 40 epochs (~6 hrs estimated on T4, fits the 9-hr Kaggle session cap)

finetune_script = '''
import os
os.chdir("/kaggle/working/sgmse")
import torch
if not hasattr(torch, "_load_patched"):
    _orig = torch.load
    def _patched(*a, **kw):
        kw["weights_only"] = False
        return _orig(*a, **kw)
    torch.load = _patched
    torch._load_patched = True

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, Callback
from sgmse.model import ScoreModel
from sgmse.data_module import SpecsDataModule

SAVE_DIR  = "/kaggle/working/sgmse_control"
DATA_DIR  = "/kaggle/working/sgmse/data"
os.makedirs(SAVE_DIR, exist_ok=True)

pl.seed_everything(42)

model = ScoreModel(
    backbone="ncsnpp",
    sde="ouve",
    data_module_cls=SpecsDataModule,
    # OUVE-SDE args
    theta=1.5, sigma_min=0.05, sigma_max=0.5, N=1000,
    # training args
    loss_type="score_matching", loss_weighting="sigma^2",
    num_eval_files=0, lr=1e-4,
    # NCSNpp backbone args
    nf=32,
    # data-module args (consumed by SpecsDataModule internally)
    base_dir=DATA_DIR, format="default", batch_size=16,
    n_fft=510, hop_length=128, num_frames=256, window="hann",
    num_workers=2, dummy=False, spec_factor=0.15, spec_abs_exponent=0.5,
    normalize="noisy", transform_type="exponent",
)
# CONTROL: no physics regularization.
model.physics_weight = 0.0

class PrintLosses(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        vl = m.get("valid_loss"); tl = m.get("train_loss_epoch")
        print("\\n[Epoch " + str(trainer.current_epoch) +
              "] train_loss=" + (str(round(float(tl),4)) if tl is not None else "?") +
              " | valid_loss=" + (str(round(float(vl),4)) if vl is not None else "?") + "\\n")

ckpt_cb = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename="control_epoch{epoch:02d}_valloss{valid_loss:.4f}",
    save_top_k=3, monitor="valid_loss", mode="min", every_n_epochs=1,
    save_last=True,
)

trainer = pl.Trainer(
    max_epochs=40, accelerator="gpu", devices=1,
    callbacks=[ckpt_cb, PrintLosses()],
    log_every_n_steps=20, enable_progress_bar=True,
    gradient_clip_val=1.0,
)
trainer.fit(model)
print("Best checkpoint:", ckpt_cb.best_model_path)
'''

with open('/kaggle/working/sgmse/finetune_control.py', 'w') as f:
    f.write(finetune_script)
print('finetune_control.py written')

finetune_control.py written


In [6]:
# Run Step 2 control. 3000 samples / bs 16 = ~188 steps/epoch × 40 epochs = ~7,500 steps.
# Estimated ~5-6 hours on Kaggle T4. Fits the 9-hour Kaggle session cap.
# Watch the printed train_loss and valid_loss — they should both decrease monotonically.
get_ipython().system('python finetune_control.py')

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-05-20 05:41:43.598628: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779255703.801629     275 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779255703.860023     275 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779255704.342577     275 computation_placer.cc:177] computation placer already registered. Please c

In [7]:
# 1. Patch enhancement.py
with open('/kaggle/working/sgmse/enhancement.py', 'r') as f:
  content = f.read()

patch = '''import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
  kwargs['weights_only'] = False
  return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

'''
if '_patched_torch_load' not in content:
  with open('/kaggle/working/sgmse/enhancement.py', 'w') as f:
      f.write(patch + content)
  print('enhancement.py patched')
else:
  print('enhancement.py already patched')

# 2. Patch Lightning's loaders (pure Python, no sed)
for path in [
  '/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py',
  '/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py',
]:
  with open(path, 'r') as f:
      src = f.read()
  new = src.replace(
      'weights_only: Optional[bool] = None,',
      'weights_only: Optional[bool] = False,',
  )
  if new != src:
      with open(path, 'w') as f:
          f.write(new)
      print('patched', path)
  else:
      print('no change needed', path)

# 3. Clear any partial output from a previous control run
import shutil, os
if os.path.exists('/kaggle/working/sgmse/enhanced_control'):
  shutil.rmtree('/kaggle/working/sgmse/enhanced_control')
  print('cleared enhanced_control/')

enhancement.py patched
patched /usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py
patched /usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py


In [8]:
# Enhancement + metrics on the trained control checkpoint.
# Runs on the full 826-sample VBD test set for direct comparison to the
# experimental run.
import glob, os
ckpts = sorted(glob.glob('/kaggle/working/sgmse_control/control_*.ckpt'))
print('checkpoints:', ckpts)
CKPT = ckpts[-1]  # most recent (= lowest val_loss epoch among saved top-3)
print('using:', CKPT)

get_ipython().system(f'python enhancement.py --test_dir data/test/noisy --enhanced_dir enhanced_control --ckpt {CKPT} --N 10')
get_ipython().system('python calc_metrics.py --clean_dir data/test/clean --noisy_dir data/test/noisy --enhanced_dir enhanced_control')

checkpoints: ['/kaggle/working/sgmse_control/control_epochepoch=17_vallossvalid_loss=1129.4205.ckpt', '/kaggle/working/sgmse_control/control_epochepoch=34_vallossvalid_loss=1245.0676.ckpt', '/kaggle/working/sgmse_control/control_epochepoch=38_vallossvalid_loss=1072.8347.ckpt']
using: /kaggle/working/sgmse_control/control_epochepoch=38_vallossvalid_loss=1072.8347.ckpt
Set TORCH_CUDA_ARCH_LIST to: 7.5;7.5
100%|█████████████████████████████████████████| 824/824 [02:37<00:00,  5.23it/s]
PESQ: 1.75 ± 0.42
ESTOI: 0.73 ± 0.13
SI-SDR: 11.7 ± 3.7
SI-SIR: 18.1 ± 5.0
SI-SAR: 13.1 ± 3.5
